In [1]:
import torch
import mic21_model
from mic21_model.configuration_mic21 import MIC21SummarizerConfig
from mic21_model.modeling_mic21 import MIC21SummarizerModel
from datasets import load_dataset
from transformers import TrainingArguments
from transformers import Trainer
import numpy as np
import pdb
from mic21_model.DistributedTrainer import DistributedTrainer

In [2]:
cuda_id = 2
device_map = torch.load('llama_3_dm.pth')
device_map['model.embed_tokens'] = 1
device_map['model.norm'] = 2
device_map['model.rotary_emb'] = 2
device_map['lm_head'] = 1
device_map['model.layers.12'] = 2
device_map['model.layers.13'] = 2
device_map['model.layers.14'] = 2
device_map['model.layers.15'] = 2
device_map['model.layers.16'] = 2
device_map['model.layers.17'] = 2
device_map['model.layers.18'] = 2
device_map['model.layers.19'] = 2
device_map['model.layers.20'] = 2
device_map['model.layers.21'] = 2
device_map['model.layers.22'] = 2
device_map['model.layers.23'] = 2
device_map['model.layers.0'] = 1
device_map['model.layers.1'] = 1
device_map['model.layers.2'] = 2
device_map['model.layers.3'] = 2

In [3]:
config = MIC21SummarizerConfig(
    device_map=device_map,
    memory_map={0: "3GiB", 1: "3GiB",2: "5GiB",},
    in_device = device_map['model.embed_tokens'],
    out_device = device_map['lm_head'],
    detectron2_cuda_id = cuda_id)
model = MIC21SummarizerModel(config)

`torch_dtype` is deprecated! Use `dtype` instead!
/usr/local/lib/python3.10/dist-packages/accelerate/utils/modeling.py:1598: UserWarning: The following device_map keys do not match any submodules in the model: ['model.layers.22', 'model.layers.23', 'model.layers.24', 'model.layers.25', 'model.layers.26', 'model.layers.27', 'model.layers.28', 'model.layers.29', 'model.layers.30', 'model.layers.31']
  warnings.warn(


In [4]:
dataset = load_dataset("mic21_dataset")

domain_name = "cricket"

indices = [i for i, label in enumerate(dataset['train']['label']) if label == domain_name]
dataset_subset = dataset['train'].select(indices)
dataset_subset = dataset_subset.remove_columns(['label','description','size','annotations'])

Using the latest cached version of the dataset since mic21_dataset couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'default' at /home/jordan/.cache/huggingface/datasets/mic21_dataset/default/0.0.0/1e7f85809b0f9e4e (last modified on Mon Nov 24 14:16:00 2025).


In [5]:
batch_size = 7
seq_len = 40
num_epoch = 100

In [6]:
def prepare_dataset(example):
    target_text = example["title"]
    target_tok = model.components["tokenizer"](target_text, add_special_tokens=False, max_length=seq_len, padding='max_length')
    img_np = np.array(example["image"])
    img_np = img_np[:, :, 2::-1].astype(np.uint8)       # Convert RGB to BGR
    example["image_bgr"] = img_np
    example["target_tok"] = target_tok
    return example

In [7]:
def data_collator_1(features):
    return {'images':[f['image'] for f in features], 'titles':[f['title'] for f in features]}

In [8]:
training_args = TrainingArguments(
    output_dir="mic21_model/train_cricket",
    learning_rate=2e-5,
    per_device_train_batch_size=3,
    per_device_eval_batch_size=3,
    num_train_epochs=100,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    push_to_hub=False,
    remove_unused_columns=False,
)

In [9]:
trainer = DistributedTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset_subset,
    eval_dataset=dataset_subset,
    data_collator=data_collator_1
)

Detected kernel version 4.15.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


In [10]:
trainer.train()

`OffloadedCache` is deprecated and will be removed in version v4.59 Use `DynamicCache(offloading=True)` instead


OutOfMemoryError: CUDA out of memory. Tried to allocate 20.00 MiB. GPU  has a total capacity of 10.90 GiB of which 17.25 MiB is free. Process 44714 has 10.88 GiB memory in use. Of the allocated memory 8.89 GiB is allocated by PyTorch, and 1.82 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [4]:
model.save_pretrained('mic21_model/test_ch')

In [10]:
model.state_dict()

OrderedDict([('projection_layer.weight',
              tensor([[ 0.0201, -0.0406, -0.0377,  ...,  0.0482,  0.0522, -0.0312],
                      [-0.0283,  0.0081, -0.0597,  ...,  0.0164,  0.0212,  0.0162],
                      [-0.0367,  0.0015,  0.0378,  ..., -0.0148, -0.0029, -0.0072],
                      ...,
                      [-0.0187,  0.0493, -0.0216,  ..., -0.0148, -0.0203,  0.0089],
                      [-0.0445, -0.0486,  0.0589,  ..., -0.0450, -0.0234, -0.0621],
                      [ 0.0429,  0.0035, -0.0071,  ...,  0.0035, -0.0305, -0.0400]],
                     device='cuda:0')),
             ('projection_layer.bias',
              tensor([-0.0058, -0.0306,  0.0281,  ..., -0.0576,  0.0414, -0.0013],
                     device='cuda:0')),
             ('projection_norm.weight',
              tensor([1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
                      1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1

In [5]:
model.push_to_hub('jkralev/mic21_model')

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

CommitInfo(commit_url='https://huggingface.co/jkralev/mic21_model/commit/0d12ceb2c8e2b550077a6f3bc333a717354af01e', commit_message='Upload model', commit_description='', oid='0d12ceb2c8e2b550077a6f3bc333a717354af01e', pr_url=None, repo_url=RepoUrl('https://huggingface.co/jkralev/mic21_model', endpoint='https://huggingface.co', repo_type='model', repo_id='jkralev/mic21_model'), pr_revision=None, pr_num=None)

In [8]:
dir(mic21_model)

['AutoConfig',
 'AutoModel',
 'MIC21SummarizerConfig',
 'MIC21SummarizerModel',
 '__builtins__',
 '__cached__',
 '__doc__',
 '__file__',
 '__loader__',
 '__name__',
 '__package__',
 '__path__',
 '__spec__',
 'configuration_mic21',
 'modeling_mic21']